# Análisis de `unique_identifier` duplicados en transacciones

## 1. Import Required Libraries

In [1]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

BASE_URL = "http://new-service:8000"


## 2. Load Transactions Data

Trae todas las transacciones desde la API paginando con `skip` / `limit`.

In [2]:
def fetch_all_transactions(base_url: str, page_size: int = 1000) -> list:
    all_transactions = []
        skip = 0
    while True:
        resp = requests.get(
            f"{base_url}/transactions/",
            params={"skip": skip, "limit": page_size},
        )
        resp.raise_for_status()
        batch = resp.json().get("transactions", [])
        all_transactions.extend(batch)
        if len(batch) < page_size:
            break
        skip += page_size
    return all_transactions

transactions = fetch_all_transactions(BASE_URL)
df = pd.DataFrame(transactions)

print(f"Total de transacciones: {len(df)}")
df.head()


IndentationError: unexpected indent (3377983945.py, line 3)

## 3. Explorar el Dataset

In [ ]:
print(df.shape)
print(df.dtypes)
print()
print("¿Existe columna 'unique_identifier'?", "unique_identifier" in df.columns)
print()
print("Muestra de unique_identifier:")
df["unique_identifier"].head(10)


## 4. Analizar `unique_identifier` duplicados

In [ ]:
total = len(df)
duplicated_mask = df["unique_identifier"].duplicated(keep=False)
n_duplicated_rows = duplicated_mask.sum()

counts = df["unique_identifier"].value_counts()
repeated_ids = counts[counts > 1]

print(f"Total filas              : {total}")
print(f"Filas con uid duplicado  : {n_duplicated_rows}  ({n_duplicated_rows/total*100:.2f}%)")
print(f"unique_identifiers únicos: {counts.shape[0]}")
print(f"unique_identifiers repet.: {len(repeated_ids)}")
print()
print("Top 20 más repetidos:")
repeated_ids.head(20)


## 5. Visualizar duplicados

In [ ]:
TOP_N = 20

if repeated_ids.empty:
    print("No hay unique_identifiers duplicados.")
else:
    top = repeated_ids.head(TOP_N)

    fig, ax = plt.subplots(figsize=(12, 5))
    sns.barplot(x=top.values, y=top.index, ax=ax, palette="Reds_r")
    ax.set_title(f"Top {TOP_N} unique_identifiers más repetidos")
    ax.set_xlabel("Cantidad de apariciones")
    ax.set_ylabel("unique_identifier")
    plt.tight_layout()
    plt.show()


## 6. Filtrar y exportar registros duplicados

In [ ]:
df_dupes = (
    df[duplicated_mask]
    .sort_values("unique_identifier")
    .reset_index(drop=True)
)

print(f"Registros con unique_identifier duplicado: {len(df_dupes)}")
df_dupes


In [ ]:
# Exportar duplicados a CSV para revisión manual
if not df_dupes.empty:
    out_path = "duplicate_unique_identifiers.csv"
    df_dupes.to_csv(out_path, index=False)
    print(f"Exportado a: {out_path}")
else:
    print("No hay duplicados que exportar.")
